# How to load and plot modes from a FELiCS computation

Before you start, you should install all necessary FELiCS packages, and install FELiCS as a package itself. The [installation guide](https://felics2-0-laboratory-for-flow-instabilities-and--112f91add91c56.gitlab-pages.tu-berlin.de/installation_guide.html) explains those steps in detail.

After installing FELiCS you can start writing your first FELiCS script. 

This tutorial explains how to load previously computed resolvent or eigenmodes from disk using the FELiCS I/O module. The results were saved in the `/Input` directory. For more information about the initial computation of modal results, see [TODO].

**1. Imports and configuration**

We begin by importing the necessary FELiCS modules and reading the simulation configuration.

In [1]:
from FELiCS.Parameters.config          import config
from FELiCS.Misc.logging               import Logger
from FELiCS.IO.Reader                  import Reader
from FELiCS.Fields.Mode                import Mode
from FELiCS.IO.Writer                  import Writer
from FELiCS.SpaceDisc.FEMSpaces        import FEMSpaces
from FELiCS.Fields.ModeCollection      import ModeCollection

🔧 Set up logging and read configuration

In [2]:
logger = Logger(True, False, "felics")
logger = Logger.get_logger("felics")

param = config()
param.importFromFile("Input/config.json")

Debug    | logging.py             | _setup_logger              (line 190 ) : Logger initialized successfully
Debug    | config.py              | __init__                   (line 46  ) : Initializing config class with defaults.
Info     | config.py              | importFromFile             (line 205 ) : Loading configuration from Input/config.json
Warning  | config.py              | importFromFile             (line 224 ) : Field "MolViscPerturbModel" missing from file, setting default: {'type': 'Constant', 'Constants': {'Viscosity': 1.0}}
Warning  | config.py              | importFromFile             (line 224 ) : Field "PrandtlNumber" missing from file, setting default: 0.72
Warning  | config.py              | importFromFile             (line 224 ) : Field "ForcingNorm" missing from file, setting default: TKE
Warning  | config.py              | importFromFile             (line 224 ) : Field "ResponseNorm" missing from file, setting default: TKE
Debug    | config.py              | impor

This creates:
- a global FELiCS logger
- a `param` object that contains all case parameters (mesh path, polynomial order, Reynolds number, etc.), loaded from `config.json`.

**2. Initialization of core FELiCS objects**

In this section we reconstruct the essential FELiCS objects exactly as they were used during the mode computation.
This ensures consistency of function spaces and mesh mappings.

In [3]:
mesh = param.getMesh()

FELiCS loads the mesh using the path and options specified in `config.json`.

In [ ]:
FEMSpaces = FEMSpaces(param, mesh)

This initializes the mixed finite-element space used for the velocity and pressure variables.
These spaces must match the ones used when the modes were originally computed.

📥 **Initialize the Reader and Writer**

In [ ]:
reader = Reader(
    sourceDir="Input",
    needInterpolation=False,
)

- The Reader searches the Input/ directory for mode files written by a previous FELiCS run.

**3. Loading modes from disk**

FELiCS stores modes in an organized modeCollection structure. To load all modes at once, we create an empty `ModeCollection` and call its `importData` method.
ModeCollection manages groups of modes, facilitates filtering, and provides utility functions (descriptions, sorting, exporting, etc.).
FELiCS will automatically:
- scan the folder
- detect available modes (direct and/or adjoint)
- reconstruct each mode as a Mode object
- rebuild all subfields in the correct function spaces
- attach metadata (type, eigenvalue, iteration number, etc.)

In [ ]:
importSolution = ModeCollection(
    FEMSpaces.VMixed,
    mesh
)
importSolution.importData(
    reader,
    "Input",
)

🔍 **Inspect loaded modes**
This prints a summary of the imported modes: mode numbers, types, eigenvalues, ...

In [ ]:
importSolution.describe()

**4. Extracting specific modes and fields**
Most analyses involve separating direct and adjoint modes, then selecting individual components of the state vector.

**🎯 Separate direct and adjoint modes**

In [ ]:
directMode  = [m for m in importSolution.modeList if m.modeType.name == "DIRECT"][0]
adjointMode = [m for m in importSolution.modeList if m.modeType.name == "ADJOINT"][0]

You can always check:

In [ ]:
print(directMode.modeType.name)

**🔬 Accessing subfields**
FELiCS organizes the mixed state vector hierarchically:

Mode
 └── Velocity field
       └── Streamwise component (u)
       └── Transverse component (v)
 └── Pressure
 └── … (other fields depending on the model)
 
 To extract the streamwise velocity:

In [ ]:
uxFieldDirect  = directMode.getListOfSubFields()[0].getListOfSubFields()[0]
uxFieldAdjoint = adjointMode.getListOfSubFields()[0].getListOfSubFields()[0]

Here:
- `getListOfSubFields()[0]` → the velocity vector
- `...getListOfSubFields()[0]` → first component, i.e. **streamwise** velocity

**5. Plotting the modes**

Each **scalar** field knows how to plot itself using Matplotlib and the triangulation utilities integrated in FELiCS.

In [ ]:
uxFieldDirect.plot(xlim=(-5, 20), ylim=(0, 5))
uxFieldAdjoint.plot(xlim=(-20, 5), ylim=(0, 5))